# Practical 2 — Large Language Models: Architecture and Practice

**Large Language Models in Finance · Chapter 2 / Lecture 2**

This notebook builds the moving parts of a transformer language model **by hand**, in
runnable code. You will implement a byte-pair-encoding (BPE) tokenizer and run it on a
small finance corpus, build a numerically stable softmax and the two decoding controls
that matter most in production — **temperature** and **top-$p$ (nucleus) sampling** — and
construct the sinusoidal **positional encoding** and probe its geometry. Every part ends
in an `assert`-based self-check, so a clean run (no `AssertionError`) means your
implementations are correct.

**Setup.** Parts 1–3 are fully offline and need only `numpy` and `matplotlib`. Part 4 is
an *optional* live call to a hosted LLM to measure real token-based API cost; it is gated
behind `RUN_API = False`, so the notebook runs end-to-end with no network and no API key.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter

rng = np.random.default_rng(0)
print("numpy", np.__version__, "ready")

## Part 1 — Byte-pair-encoding tokenization, by hand  `[B]`

Production LLM tokenizers (GPT, Llama, Claude) are learned with **byte-pair encoding**:
start from single characters, then repeatedly merge the most frequent adjacent pair into a
new symbol. Frequent finance morphemes like `trad`, `ing`, `er` become single tokens, so a
word like `trading` costs fewer tokens than its character count — which is exactly what
drives context-window usage and API cost.

**Your task:** complete `learn_bpe` (the merge loop) and `tokenize_word` (apply the learned
merges left-to-right). The `</w>` marker denotes an end-of-word boundary.

In [ ]:
CORPUS = ("trader trader trader trading trading trading trade trade trades "
          "lower lower lowest slower slowest newer newest").split()
NUM_MERGES = 12

def get_vocab(corpus):
    # each word -> space-separated characters plus an end-of-word marker
    vocab = Counter()
    for w in corpus:
        vocab[" ".join(list(w)) + " </w>"] += 1
    return vocab

def get_pair_counts(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        sym = word.split()
        for i in range(len(sym) - 1):
            pairs[(sym[i], sym[i + 1])] += freq
    return pairs

def merge_pair(pair, vocab):
    # robust merge: only join the two symbols where they are adjacent tokens
    pat = re.compile(r"(?<!\S)" + re.escape(" ".join(pair)) + r"(?!\S)")
    joined = "".join(pair)
    return {pat.sub(joined, word): freq for word, freq in vocab.items()}

def learn_bpe(corpus, num_merges):
    vocab = get_vocab(corpus)
    merges = []
    # TODO: each round, count pairs, take the most frequent, record + apply the merge
    for _ in range(num_merges):
        pairs = get_pair_counts(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)
        vocab = merge_pair(best, vocab)
    return merges

def tokenize_word(word, merges):
    sym = list(word) + ["</w>"]
    # TODO: apply each learned merge in order, joining adjacent (a, b) -> "ab"
    for a, b in merges:
        out, i = [], 0
        while i < len(sym):
            if i < len(sym) - 1 and sym[i] == a and sym[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(sym[i]); i += 1
        sym = out
    return sym

merges = learn_bpe(CORPUS, NUM_MERGES)
print("learned merges:", [a + b for a, b in merges])
for w in ["trading", "trader", "slowest"]:
    print(f"  {w:>8} -> {tokenize_word(w, merges)}")

In [ ]:
# self-check (Part 1)
assert len(merges) == NUM_MERGES, "expected NUM_MERGES merges from this corpus"

# every word must round-trip: concatenating its tokens (minus the marker) rebuilds it
for w in set(CORPUS):
    toks = tokenize_word(w, merges)
    assert "".join(toks).replace("</w>", "") == w, (w, toks)

# BPE must compress: 'trading' should take fewer tokens than its 7 chars + marker (=8)
assert len(tokenize_word("trading", merges)) < len("trading") + 1

# the merges actually fire on a held-out, never-seen word built from learned pieces
assert len(tokenize_word("slower", merges)) < len("slower") + 1

# total corpus token count is strictly below the raw character+marker baseline
raw = sum(len(w) + 1 for w in CORPUS)
bpe = sum(len(tokenize_word(w, merges)) for w in CORPUS)
assert bpe < raw, (bpe, raw)
print(f"Part 1 OK  (corpus: {raw} char-tokens -> {bpe} BPE tokens)")

## Part 2 — Softmax, temperature, and top-$p$ sampling  `[I]`

At each step a language model emits a logit vector over the vocabulary. A **softmax** turns
it into a probability distribution; **temperature** $\tau$ rescales the logits before the
softmax ($\tau<1$ sharpens, $\tau>1$ flattens); **top-$p$ (nucleus) sampling** keeps only the
smallest set of most-probable tokens whose cumulative mass reaches $p$, then renormalizes —
truncating the unreliable tail.

**Your task:** complete `softmax` (row-safe, numerically stable), `temperature_softmax`, and
`top_p_filter`. The self-check verifies that temperature raises entropy and that top-$p$
truncates to the correct minimal nucleus.

In [ ]:
# eight candidate next tokens for a finance prompt "...quarterly revenue ___"
TOKENS = ["rose", "grew", "fell", "beat", "missed", "surged", "declined", "stagnated"]
LOGITS = np.array([3.4, 2.1, 1.2, 0.9, 0.3, -0.4, -1.1, -2.0])

def softmax(z):
    # TODO: subtract the max for stability, exponentiate, normalize (last axis)
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def temperature_softmax(logits, tau):
    # TODO: divide logits by tau before the softmax
    return softmax(logits / tau)

def entropy(p):
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def top_p_filter(probs, p):
    order = np.argsort(probs)[::-1]                 # indices, high prob first
    cum = np.cumsum(probs[order])
    # TODO: keep tokens up to and including the first that reaches cumulative p
    k = int(np.searchsorted(cum, p) + 1)
    keep = order[:k]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()

base = softmax(LOGITS)
print("greedy pick:", TOKENS[int(base.argmax())])
print("H(tau=0.5)=%.3f  H(tau=1)=%.3f  H(tau=2)=%.3f"
      % (entropy(temperature_softmax(LOGITS, 0.5)),
         entropy(temperature_softmax(LOGITS, 1.0)),
         entropy(temperature_softmax(LOGITS, 2.0))))

filt = top_p_filter(base, 0.9)
kept = [TOKENS[i] for i in np.where(filt > 0)[0]]
print("top-p=0.9 nucleus keeps:", kept)

# draw a few samples from the truncated distribution to show it is usable
draws = rng.choice(len(TOKENS), size=8, p=filt)
print("samples from nucleus:", [TOKENS[i] for i in draws])

In [ ]:
# self-check (Part 2)
# softmax is a valid distribution and matches a hand reference
assert np.isclose(base.sum(), 1.0) and (base >= 0).all()
ref = np.exp([1.0, 2.0, 3.0]); ref = ref / ref.sum()
assert np.allclose(softmax(np.array([1.0, 2.0, 3.0])), ref)

# temperature is monotone in entropy: hotter -> more uncertain
h = [entropy(temperature_softmax(LOGITS, t)) for t in (0.5, 1.0, 2.0)]
assert h[0] < h[1] < h[2], h

# top-p truncates: it drops at least one token and stays a valid distribution
nkeep = int((filt > 0).sum())
assert nkeep < len(TOKENS), "top-p must remove some tail mass"
assert np.isclose(filt.sum(), 1.0)

# and it keeps the *minimal* nucleus: cum mass of the kept set >= p,
# but removing its least-likely member would drop below p
order = np.argsort(base)[::-1]
cum = np.cumsum(base[order])
assert cum[nkeep - 1] >= 0.9 - 1e-12
assert nkeep == 1 or cum[nkeep - 2] < 0.9

# the single lowest-probability token is always excluded at p=0.9
assert filt[int(base.argmin())] == 0.0
print(f"Part 2 OK  (top-p=0.9 nucleus size = {nkeep}/{len(TOKENS)})")

## Part 3 — Sinusoidal positional encoding  `[A]`

Self-attention is permutation-equivariant: on its own it cannot tell token 1 from token 50.
The original transformer injects order with a fixed **sinusoidal positional encoding**
(Definition in the chapter):

$$\mathrm{PE}[pos, 2i] = \sin\!\Big(\tfrac{pos}{10000^{2i/d}}\Big), \qquad
  \mathrm{PE}[pos, 2i+1] = \cos\!\Big(\tfrac{pos}{10000^{2i/d}}\Big).$$

**Your task:** complete `positional_encoding`. The self-check verifies the matrix is bounded,
has all-distinct rows (every position is uniquely encoded), and that cosine similarity
**decays** with positional distance — the same property the chapter figure illustrates.

In [ ]:
D_MODEL, SEQ_LEN = 64, 50

def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    # TODO: angle rate 1 / 10000**(2*(i//2)/d_model); sin on even cols, cos on odd
    rates = 1.0 / np.power(10000.0, (2 * (i // 2)) / d_model)
    ang = pos * rates
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(ang[:, 0::2])
    pe[:, 1::2] = np.cos(ang[:, 1::2])
    return pe

def cos_sim(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

PE = positional_encoding(SEQ_LEN, D_MODEL)
print("PE shape:", PE.shape, " range: [%.3f, %.3f]" % (PE.min(), PE.max()))
print("cos(PE_5, PE_10) = %.3f   cos(PE_5, PE_45) = %.3f"
      % (cos_sim(PE[5], PE[10]), cos_sim(PE[5], PE[45])))

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(PE.T, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
ax.set_xlabel("Token position"); ax.set_ylabel("Encoding dimension")
ax.set_title(r"Sinusoidal positional encoding ($d_\mathrm{model}=64$, $n=50$)")
fig.colorbar(im, ax=ax, label="value"); plt.tight_layout(); plt.show()

In [ ]:
# self-check (Part 3)
assert PE.shape == (SEQ_LEN, D_MODEL)
assert PE.min() >= -1.0 - 1e-9 and PE.max() <= 1.0 + 1e-9, "sin/cos are bounded in [-1, 1]"

# every position has a distinct encoding row (no collisions)
assert len({tuple(np.round(r, 6)) for r in PE}) == SEQ_LEN

# position 0: even dims sin(0)=0, odd dims cos(0)=1
assert np.allclose(PE[0, 0::2], 0.0) and np.allclose(PE[0, 1::2], 1.0)

# cosine similarity decays with positional separation (chapter's claim)
assert cos_sim(PE[5], PE[10]) > cos_sim(PE[5], PE[45])
print("Part 3 OK")

## Part 4 — Real API cost, measured  *(optional, requires internet + API key)*  `[I]`

The earlier parts are free and local. A hosted LLM bills per **token**, split into input and
output. This cell makes one real call, reads the exact token usage off the response, and
converts it to dollars using the model's published rate — the concrete version of the
"context window costs money" point from the chapter. It is gated so the notebook stays fully
offline by default.

```bash
pip install anthropic
export ANTHROPIC_API_KEY='sk-ant-...'
```

In [ ]:
RUN_API = False   # set True (with anthropic installed and ANTHROPIC_API_KEY set) to run

# Published per-million-token pricing for the model below (USD).
MODEL = "claude-opus-4-8"
PRICE_IN_PER_MTOK, PRICE_OUT_PER_MTOK = 5.00, 25.00

if RUN_API:
    from anthropic import Anthropic
    client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    resp = client.messages.create(
        model=MODEL,
        max_tokens=200,
        messages=[{"role": "user",
                   "content": "In one sentence, what is byte-pair encoding?"}],
    )
    text = "".join(b.text for b in resp.content if b.type == "text")
    n_in, n_out = resp.usage.input_tokens, resp.usage.output_tokens
    cost = n_in / 1e6 * PRICE_IN_PER_MTOK + n_out / 1e6 * PRICE_OUT_PER_MTOK
    print(text)
    print(f"\ntokens: {n_in} in + {n_out} out -> ${cost:.6f} for this call")
else:
    # Offline cost model: estimate the bill for a hypothetical 10-K analysis run
    # (~128k input tokens, 2k output) using the same arithmetic, no network.
    n_in, n_out = 128_000, 2_000
    cost = n_in / 1e6 * PRICE_IN_PER_MTOK + n_out / 1e6 * PRICE_OUT_PER_MTOK
    print("RUN_API is False — offline cost estimate for a 128k-token 10-K pass:")
    print(f"  {n_in} in + {n_out} out -> ${cost:.4f} at {MODEL} rates")

In [ ]:
# self-check (Part 4) — the cost arithmetic is correct whether or not the call ran
expected = n_in / 1e6 * PRICE_IN_PER_MTOK + n_out / 1e6 * PRICE_OUT_PER_MTOK
assert abs(cost - expected) < 1e-12
assert cost > 0 and n_in > 0
print("Part 4 OK")

## What you built

- A **byte-pair-encoding tokenizer** — learned merges on a finance corpus and verified it
  both round-trips and compresses (`Part 1`)
- A numerically stable **softmax** plus **temperature** and **top-$p$** decoding, with a
  check that the nucleus truncates to the correct minimal set (`Part 2`)
- **Sinusoidal positional encoding** from scratch, with a geometric self-check that
  similarity decays with distance (`Part 3`)
- A **token-based API cost** calculation, runnable live or estimated offline (`Part 4`)